##### Copyright 2026 Google LLC.

In [6]:
# @title Licensed under the Apache License, Version 2.0 (the "License");
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Gemini API: Get started with Lyria 3.5 🎵

<a target="_blank" href="https://colab.research.google.com/github/google-gemini/cookbook/blob/main/quickstarts/Get_started_Lyria.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" height=30/></a>

This notebook will show you how to generate high-fidelity music with **Lyria 3.5**, Google DeepMind's advanced music generation model.

Whether you need a short musical loop or a full-length 3-minute song with rich vocals and structural complexity, Lyria 3.5 makes it possible with the simple `interactions.create` call.

In this notebook, you will learn how to:
- Generate music from text prompts.
- Create "visual soundtracks" where an image serves as the inspiration.
- Use advanced timestamped prompts to control song structure.
- Generate purely instrumental tracks.
- Create vocal tracks in multiple languages.

This guide will walk you through generating your first hit song using the GenAI SDK and the Interactions API.

> **Note:** This notebook uses the [Interactions API](https://ai.google.dev/gemini-api/docs/interactions), the latest way to interact with Gemini models. Looking for the `generateContent` version? Check the [archive branch](https://github.com/google-gemini/cookbook/blob/archive/generate-content-api/quickstarts/Get_started_Lyria.ipynb).

<a name="setup"></a>
## Setup

### Install SDK

Install the SDK from [PyPI](https://github.com/googleapis/python-genai). It's recommended to always use the latest version.

In [7]:
%pip install -U -q "google-genai>=2.9.0" # 1.62 is needed for music generation

### Setup your API key

To run the following cell, your API key must be stored in a Colab Secret named `GEMINI_API_KEY`. If you don't already have an API key or you aren't sure how to create a Colab Secret, see [Authentication ![image](https://storage.googleapis.com/generativeai-downloads/images/colab_icon16.png)](../quickstarts/Authentication.ipynb) for a walkthrough.

In [9]:
from google.colab import userdata

GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')

SecretNotFoundError: Secret GEMINI_API_KEY does not exist.

### Initialize SDK client

With the new SDK, now you only need to initialize a client with your API key (or OAuth if using [Vertex AI](https://cloud.google.com/vertex-ai)). The model is now set in each call.

In [ ]:
from google import genai
from google.genai import types

client = genai.Client(api_key=GEMINI_API_KEY)

## Select a model

Lyria 3.5 is our advanced, full-song generative model with enhanced audio fidelity and vocal clarity, optimized for precise prompt adherence and rich, cohesive musical arrangements (generating up to 3-minute songs with intro, verse, chorus!). 🎼

*   `lyria-3.5`: Our flagship music generation model.
*   `lyria-3-pro-preview`: Generates full songs (up to 3 minutes) with structure (intro, verse, chorus!).
*   `lyria-3-clip-preview`: Perfect for quick 30-second clips, loops, and experiments.

More details in the [documentation](https://ai.google.dev/gemini-api/docs/music-generation).

In [ ]:
# @title Select a model
MODEL_ID = "lyria-3.5"  # @param ["lyria-3.5", "lyria-3-pro-preview", "lyria-3-clip-preview"] {"allow-input":true, isTemplate: true}


## Generate your first song

Generating a song is straightforward using the `interactions.create` call. You MUST specify the `Audio` and `Text` modalities in the configuration to receive the audio data and the generated lyrics/structure.

In [ ]:
prompt = "Create a 30-second cheerful acoustic folk song about a sunrise in the mountains."  # @param {type:"string"}

interaction = client.interactions.create(
    model=MODEL_ID,
    input=prompt,
    response_modalities=["audio", "text"],  # To get both lyrics and the song, this is the default setting
)

### Parsing the output

The output will contain both text and audio parts:

In [ ]:
for step in interaction.steps:
    print(step)

The first part always the lyrics:

In [ ]:
# Lyrics
for step in interaction.steps:
    if step.type == 'model_output' and hasattr(step, 'content'):
        for content in step.content:
            if hasattr(content, 'text') and content.text:
                print(content.text)

The last part's `inline_data` contains the audio file in base64:

In [ ]:
from IPython.display import display, Audio
import base64

for step in interaction.steps:
    if step.type == 'model_output' and hasattr(step, 'content'):
        for content in step.content:
            if hasattr(content, 'data') and content.data:
                audio_bytes = base64.b64decode(content.data)
                display(Audio(data=audio_bytes, rate=48000))

#### Util

Let's now create a utility function to do all of that parsing at all:

In [ ]:
from IPython.display import display, Audio
import json
import base64

def display_lyria_response(interaction):
    metadata = {}
    lyrics_text = None
    meta_text = None

    # Collect text and audio from interaction steps
    text_parts = []
    audio_data = None

    for step in interaction.steps:
        if step.type == 'text' and hasattr(step, 'text') and step.text:
            text_parts.append(step.text)
        elif step.type == 'audio' and hasattr(step, 'data') and step.data:
            audio_data = step.data
        elif step.type == 'model_output' and hasattr(step, 'content'):
            for content in step.content:
                if hasattr(content, 'text') and content.text:
                    text_parts.append(content.text)
                elif hasattr(content, 'data') and content.data:
                    audio_data = content.data

    # 1st text part is lyrics/structure
    if len(text_parts) >= 1:
        lyrics_text = text_parts[0]

    # 2nd text part is metadata JSON
    if len(text_parts) >= 2:
        meta_text = text_parts[1]
        try:
            metadata = json.loads(meta_text)
        except json.JSONDecodeError:
            metadata = {'raw': meta_text}

    if lyrics_text:
        print('🎤 Lyrics / Structure:')
        print(lyrics_text)
        print()

    if metadata:
        print('📋 Metadata:')
        print(json.dumps(metadata, indent=2))
        print()

    # Play the audio
    if audio_data:
        print('🔊 Playing generated audio:')
        display(Audio(data=base64.b64decode(audio_data), rate=48000))

In [ ]:
display_lyria_response(interaction)

## Visual inspiration: Image-to-Music 🖼️➡️🎵

Lyria 3.5 can "invent" a song from a visual cue. Passing an image (actually up to 10 images) in the `contents` list gives the model a whole new dimension of creativity. Let's see how a grocery list can inspire a song.

First, download a test image:

In [ ]:
import base64
from io import BytesIO
from PIL import Image
import requests

# Download an example image
url = 'https://storage.googleapis.com/generativeai-downloads/images/groceries.jpeg'
image_response = requests.get(url)
image = Image.open(BytesIO(image_response.content))
b64_image = base64.b64encode(image_response.content).decode('utf-8')

# Generate music inspired by the image
interaction = client.interactions.create(
    model=MODEL_ID,
    input=[
        {
            'type': 'text',
            'text': (
                'An epic song with opera voices about this quest. Deep synths'
                ' and a speeding up tempo.'
            ),
        },
        {'type': 'image', 'data': b64_image, 'mime_type': 'image/jpeg'},
    ],
)

# Display image and audio
display(image)
display_lyria_response(interaction)

<a name="generatecontent"></a>

## Using the generateContent API

You can also use the `generateContent` API directly. Here's a quick example:

```python
response = client.models.generate_content(
    model=MODEL_ID,
    contents=prompt,
    config=types.GenerateContentConfig(
        response_modalities=['Audio', 'Text']
    )
)
```

The response structure is slightly different — check the [generateContent notebook](./Get_started_Generate_Content.ipynb) for details.

## Priompting tips

### BPM (Beats-per-minute)

Tell Lyria 3.5 which BPM you want according to the style and the rhythm you want, from very slow jazz song to very fast German techno.

In [ ]:
prompt = """
  Create a hyper-energetic Geman techno track at 180 BPM. Feature a Risset accelerando illusion with overlapping, accelerating kick drums and industrial synths that sound like they are endlessly speeding up into chaotic infinity.
"""

interaction = client.interactions.create(
    model=MODEL_ID,
    input=prompt,
)

display_lyria_response(interaction)

### Timing ⏱

You can specify exactly what happens at specific moments in the song using timestamps. This is useful for distributing lyrics or controlling instrumental shifts as the composition develops.

Let's tell the model what to do in each 10-second block of a song:

In [ ]:
prompt = """
  [0:00 - 0:10] Fast acoustic guitar arpeggios, setting an energetic tone.
  [0:10 - 0:20] Add a warm Fender Rhodes piano melody.
  [0:20 - 0:30] Full band with upbeat drums and soaring synth leads.
"""

interaction = client.interactions.create(
    model=MODEL_ID,
    input=prompt,
)

display_lyria_response(interaction)

### Music structure

You can also give the song structure using `[Intro]`, `[Verse]`, `[Chorus]`, `[Bridge]`, `[Outro]`:

In [ ]:
prompt = """
  [Intro] Calm piano music setting a sunset scene on the beach
  [Verse] Epic rock balade as the storm rages.
  [Outro] Opera with choir as the sun reappears again through the black clouds .
"""

interaction = client.interactions.create(
    model=MODEL_ID,
    input=prompt,
)

display_lyria_response(interaction)

### Control the intensity and the scale of each part of the song (full example)

Not only can you prompt each part separately, but you can also set the intensite and the scale for each of them. Here's a full example of a complex prompt:

In [ ]:
prompt = """
  [0:00 - 0:12] Intro: Begin atmospherically with just the Fender Rhodes playing
  soft chords (Cm7, Fm7). Drench it in warm reverb and introduce a light
  atmospheric texture. The mood is hazy, like a memory coming into focus.
  Intensity: 1/10 (Very Low)

  [0:12 - 0:24] Verse 1: The laid-back drum beat enters with a simple kick and
  snare. A soft, ethereal synth pad swells in the background. A clean, subtle
  sub-bass joins, adding depth. The Rhodes melody becomes slightly more defined
  over a | Cm7 | Fm7 | G7alt | Cm7 | progression. Intensity: 3/10 (Low)

  [0:24 - 0:36] Build: The groove deepens as a gentle, syncopated hi-hat is
  added. A simple, memorable lead melody appears, played on a warm, rounded
  synth. This section should feel like the gentle peak of the track's focus,
  flowing through an | Abmaj7 | G7alt | Cm7 | Ebmaj7 | progression. Intensity:
  5/10 (Medium)

  [0:36 - 0:48] Chorus: Gracefully pull back the intensity. The synth lead
  melody fades out, returning focus to the core Rhodes groove and the drums.
  This gives the track space to breathe over an | Fm7 | Ebmaj7 | Abmaj7 | G7alt
  | progression. Intensity: 4/10 (Medium-Low)

  [0:48 - 1:00] Outro: The drums and bass drop out completely. The track fades
  out leaving only the Rhodes playing spacious chords, the lingering synth pad,
  and the persistent atmospheric texture. Intensity: 2/10 (Very Low)

"""

interaction = client.interactions.create(
    model=MODEL_ID,
    input=prompt,
)

display_lyria_response(interaction)

## Control the lyrics

### Provide the lyrics

You can provide the lyrics you want in the prompt. Just make sure to clearly stat what's lyrics and what's prompt for the music.

In [ ]:
prompt = """
  An uplifting song with guitar rifts about nano banana.
  The lyrics should be:
    Yellow peel, a tiny sweet, The Nano Banana, a tropical treat. But wait—it
    hums, it starts to create, Switching into AI mode. Not a fruit, but a smart
    machine, The bananiest model you've ever seen.
"""

interaction = client.interactions.create(
    model=MODEL_ID,
    input=prompt,
)

display_lyria_response(interaction)

### Use the model reasoning capabilities to come up with lyrics

You can also let the model come up with lyrics about all kind of topics:

In [ ]:
prompt = "A serious moment in a Shakespeare play"  # @param {type:"string"}

interaction = client.interactions.create(
    model=MODEL_ID,
    input=prompt,
    response_modalities=["audio", "text"],
)

display_lyria_response(interaction)

### Create songs in different languages 🌍🎤

Lyria 3.5 generates lyrics in the language it sees in your prompt. By providing instructions in French, Spanish, or Japanese, the model will produce a vocal performance that matches the pronunciation and emotional tone of that language. Let's create a disco track in Spanish and French:

In [ ]:
prompt = "Tell me how bubble sort works as a disco song in spanish and french." # @param {type:"string"}

interaction = client.interactions.create(
    model=MODEL_ID,
    input=prompt,
)

display_lyria_response(interaction)

## Instrumental-only generation 🎹

For background music, soundtracks, or game loops where vocals aren't needed, you can use the `instrumental_only` parameter. This is faster than adding "instrumental" to your prompt and ensures a vocal-free track.

In [ ]:
prompt = "Create a looping medidation music that feels like the wind." # @param {type:"string"}

interaction = client.interactions.create(
    model=MODEL_ID,
    input=prompt,
)

display_lyria_response(interaction)

## What's next?

- Listen to your creations! 🎧
- Explore [Google AI Studio](https://aistudio.google.com/new_music) to iterate on your prompts visually.
- Check those cool AI Studio apps: [Lyria Studio](https://aistudio.google.com/apps/bundled/lyria_studio) where you can create songs wand generate karaoke versions of them and [Lyria rhythm](https://aistudio.google.com/apps/bundled/lyria_rhythm) that create custom song for a rhythm game.
- Combine music generation with [image generation](https://ai.google.dev/gemini-api/docs/image-generation) for full multimedia content.
- Dive into the [Lyria technical report](https://deepmind.google/technologies/lyria/) to learn about the architecture behind these models.